# Hafta 8 — Uçtan Uca Kurulan Transformer

Her şeyi değiştiren mimari. Vaswani vd. (2017) Transformer'ının tamamını — çok başlı dikkat, konumsal kodlama, artık bağlantılar, katman normalizasyonu, hepsini — `nn.Transformer` *kullanmadan* PyTorch ile uyguluyoruz. Uygulamamızı PyTorch'un yerleşik sürümüyle doğruluk açısından karşılaştırmak 4. alıştırmanın konusudur.

## Öğrenme Hedefleri

- Ölçekli nokta çarpımı dikkatini türetmek ve $\sqrt{d_k}$ ölçeklemesini açıklamak.
- Çok başlı dikkati PyTorch ile sıfırdan uygulamak.
- Sinüzoidal konumsal kodlamayı uygulamak ve modern alternatifler olarak RoPE ile ALiBi'yi tartışmak.
- Tam bir kodlayıcı–kod çözücü Transformer'ı bir araya getirip eğitmek.

## Zorunlu Okumalar

- Vaswani, A. vd. (2017). *Attention Is All You Need*.
- Su, J. vd. (2021). *RoFormer: Enhanced Transformer with Rotary Position Embedding*.
- Press, O., Smith, N. A. ve Lewis, M. (2022). *Train Short, Test Long: Attention with Linear Biases*.

In [ ]:
import sys, math
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path('..').resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import torch
import torch.nn as nn
import torch.nn.functional as F
torch.manual_seed(0); np.random.seed(0)

## 1. Ölçekli nokta çarpımı dikkati

$$\text{Attn}(Q, K, V) = \text{softmax}\!\left(\frac{Q K^\top}{\sqrt{d_k}}\right) V.$$

**Neden $\sqrt{d_k}$?** $Q$ ve $K$'nin girdileri birim varyanslıysa $(QK^\top)_{ij}$'nin varyansı $d_k$'dir. Ölçekleme olmadan büyük $d_k$, softmax'ı gradyanların kaybolduğu doygunluk bölgesine iter. $\sqrt{d_k}$'ye bölmek birim varyansı geri getirir ve softmax'ı bilgi taşıyan rejiminde tutar.

In [ ]:
def scaled_dot_product_attention(Q, K, V, mask=None):
    # Q, K, V: (..., T, d)
    d_k = Q.size(-1)
    scores = Q @ K.transpose(-2, -1) / math.sqrt(d_k)
    if mask is not None:
        scores = scores.masked_fill(mask, float('-inf'))
    weights = F.softmax(scores, dim=-1)
    return weights @ V, weights

# Akıl sağlığı denetimi: dikkat çıktısı değer boyutunu korur, ağırlıkların toplamı 1'dir.
Q = torch.randn(2, 5, 8); K = torch.randn(2, 5, 8); V = torch.randn(2, 5, 8)
out, w = scaled_dot_product_attention(Q, K, V)
print(f"çıktı boyutu: {tuple(out.shape)};  ağırlıkların toplamı 1: {torch.allclose(w.sum(-1), torch.ones(2, 5))}")

## 2. Çok başlı dikkat

$Q, K, V$'yi $d_k = d_{\text{model}}/h$ boyutlu $h$ alt uzaya izdüşür, her birinde ölçekli nokta çarpımı dikkati çalıştır, birleştir ve geri izdüşür.

$$\text{MHA}(X) = [\text{head}_1, \ldots, \text{head}_h] W_O, \quad \text{head}_i = \text{Attn}(X W_Q^{(i)}, X W_K^{(i)}, X W_V^{(i)}).$$

Bunu başlar üzerinde bir Python döngüsüyle değil, tek bir yığınlanmış matris çarpımıyla uyguluyoruz.

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()
        assert d_model % n_heads == 0
        self.d_model, self.n_heads = d_model, n_heads
        self.d_k = d_model // n_heads
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)

    def forward(self, q, k, v, mask=None):
        B, Tq, _ = q.shape
        Tk = k.size(1)
        Q = self.W_q(q).view(B, Tq, self.n_heads, self.d_k).transpose(1, 2)
        K = self.W_k(k).view(B, Tk, self.n_heads, self.d_k).transpose(1, 2)
        V = self.W_v(v).view(B, Tk, self.n_heads, self.d_k).transpose(1, 2)
        # maske başlar üzerinde yayınlanır: (B, 1, Tq, Tk) veya (1, 1, Tq, Tk)
        if mask is not None and mask.dim() == 2:
            mask = mask.unsqueeze(0).unsqueeze(0)
        out, _ = scaled_dot_product_attention(Q, K, V, mask)
        out = out.transpose(1, 2).contiguous().view(B, Tq, self.d_model)
        return self.W_o(out)

mha = MultiHeadAttention(d_model=64, n_heads=8)
x = torch.randn(2, 10, 64)
print(f"MHA çıktısı: {tuple(mha(x, x, x).shape)} ((2, 10, 64) olmalı)")

## 3. Konumsal kodlama

Öz-dikkat permütasyona eşdeğişkendir — kendi başına konum kavramı yoktur. Orijinal Transformer **sinüzoidal** konum vektörleri ekler:

$$\text{PE}_{(\text{pos}, 2i)} = \sin(\text{pos}/10000^{2i/d}), \quad \text{PE}_{(\text{pos}, 2i+1)} = \cos(\text{pos}/10000^{2i/d}).$$

Temel özellik: herhangi bir kaydırma $\text{PE}_{\text{pos}+k}$, $\text{PE}_{\text{pos}}$'un doğrusal bir fonksiyonudur; bu yüzden kodlama mutlak olsa da model göreli konumları öğrenebilir.

**Modern alternatifler.**

- **RoPE** (Su vd., 2021): $Q$ ve $K$'yi konuma bağlı bir açıyla döndür.
- **ALiBi** (Press vd., 2022): hiç konumsal gömme kullanmadan dikkat skorlarına $-|i - j|$ ile orantılı sabit bir sapma ekle. Eğitimde görülenden daha uzun dizilere genellenir.

In [ ]:
class SinusoidalPE(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(max_len, dtype=torch.float).unsqueeze(1)
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer('pe', pe)

    def forward(self, x):
        return x + self.pe[:x.size(1)]

pe = SinusoidalPE(d_model=64, max_len=100).pe.numpy()
fig, ax = plt.subplots(figsize=(10, 4))
im = ax.imshow(pe.T, aspect='auto', cmap='RdBu')
ax.set(xlabel='konum', ylabel='boyut', title='Sinüzoidal konumsal kodlama (64 boyut, 100 konum)')
plt.colorbar(im, ax=ax); plt.tight_layout(); plt.show()

## 4. Tam Transformer bloğu

Ön-normalizasyon (pre-norm) varyantı (orijinal son-normalizasyona göre eğitimi daha kararlıdır):

$$\mathbf{x} \leftarrow \mathbf{x} + \text{MHA}(\text{LN}(\mathbf{x})), \qquad \mathbf{x} \leftarrow \mathbf{x} + \text{FFN}(\text{LN}(\mathbf{x})).$$

FFN genişleme oranı geleneksel olarak 4×'tür (yani gizli boyut = $4 d_{\text{model}}$).

In [ ]:
class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.fc1 = nn.Linear(d_model, d_ff)
        self.fc2 = nn.Linear(d_ff, d_model)

    def forward(self, x):
        return self.fc2(F.gelu(self.fc1(x)))

class EncoderBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout=0.1):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = MultiHeadAttention(d_model, n_heads)
        self.ln2 = nn.LayerNorm(d_model)
        self.ffn = FeedForward(d_model, d_ff)
        self.drop = nn.Dropout(dropout)

    def forward(self, x, src_mask=None):
        x = x + self.drop(self.attn(self.ln1(x), self.ln1(x), self.ln1(x), src_mask))
        x = x + self.drop(self.ffn(self.ln2(x)))
        return x

class DecoderBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout=0.1):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.self_attn = MultiHeadAttention(d_model, n_heads)
        self.ln2 = nn.LayerNorm(d_model)
        self.cross_attn = MultiHeadAttention(d_model, n_heads)
        self.ln3 = nn.LayerNorm(d_model)
        self.ffn = FeedForward(d_model, d_ff)
        self.drop = nn.Dropout(dropout)

    def forward(self, x, memory, tgt_mask=None, mem_mask=None):
        h = self.ln1(x)
        x = x + self.drop(self.self_attn(h, h, h, tgt_mask))
        h = self.ln2(x)
        x = x + self.drop(self.cross_attn(h, memory, memory, mem_mask))
        x = x + self.drop(self.ffn(self.ln3(x)))
        return x

class Transformer(nn.Module):
    def __init__(self, V, d_model=64, n_heads=4, d_ff=256, n_layers=2, max_len=64, pad=0):
        super().__init__()
        self.pad = pad
        self.src_emb = nn.Embedding(V, d_model, padding_idx=pad)
        self.tgt_emb = nn.Embedding(V, d_model, padding_idx=pad)
        self.pe = SinusoidalPE(d_model, max_len)
        self.encoder = nn.ModuleList([EncoderBlock(d_model, n_heads, d_ff) for _ in range(n_layers)])
        self.decoder = nn.ModuleList([DecoderBlock(d_model, n_heads, d_ff) for _ in range(n_layers)])
        self.ln_out = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, V)

    def _make_pad_mask(self, x):
        return (x == self.pad).unsqueeze(1).unsqueeze(2)  # (B, 1, 1, T)

    def _make_causal_mask(self, T, device):
        return torch.triu(torch.ones(T, T, dtype=torch.bool, device=device), diagonal=1)

    def forward(self, src, tgt):
        src_pad = self._make_pad_mask(src)
        tgt_pad = self._make_pad_mask(tgt)
        causal = self._make_causal_mask(tgt.size(1), tgt.device)
        tgt_mask = tgt_pad | causal  # yayınlama (broadcast)

        x = self.pe(self.src_emb(src))
        for layer in self.encoder:
            x = layer(x, src_pad)
        memory = x

        y = self.pe(self.tgt_emb(tgt))
        for layer in self.decoder:
            y = layer(y, memory, tgt_mask, src_pad)
        return self.head(self.ln_out(y))

## 5. Rakam tersine çevirme görevinde eğitim

7. haftadaki rakam tersine çevirme görevini yeniden kullanıyoruz — ama bu kez bir Transformer ile. Transformer'ı kararlı eğitmek için üç standart uygulama kullanıyoruz:

- **Öğrenme oranı ısınması + kosinüs azalması** (Vaswani vd., 2017 de ısınma kullanır): ilk adımlarda Adam'ın moment tahminleri henüz güvenilir değildir.
- **Gradyan kırpma** (norm ≤ 1).
- **Değerlendirmede `model.eval()`**: dropout yalnızca eğitimde açık olmalıdır.

Model 4–7 uzunluklarında eğitilir; 8 ve 10, eğitimde hiç görülmemiş uzunluklardır.

> Bu hücrenin CPU'da çalışması yaklaşık 40 saniye sürer.

In [ ]:
DIGITS = ['zero','one','two','three','four','five','six','seven','eight','nine']
SPECIALS = ['<pad>', '<s>', '</s>']
VOCAB = SPECIALS + DIGITS
tok2id = {t: i for i, t in enumerate(VOCAB)}
id2tok = {i: t for t, i in tok2id.items()}
V = len(VOCAB)
PAD, BOS, EOS = tok2id['<pad>'], tok2id['<s>'], tok2id['</s>']

def make_batch(N, length, max_len=12):
    src = torch.full((N, max_len), PAD)
    tgt = torch.full((N, max_len + 2), PAD)
    for i in range(N):
        digits = np.random.randint(0, 10, size=length)
        src[i, :length] = torch.tensor([tok2id[DIGITS[d]] for d in digits])
        tgt[i, 0] = BOS
        for j, d in enumerate(digits[::-1]):
            tgt[i, j + 1] = tok2id[DIGITS[d]]
        tgt[i, length + 1] = EOS
    return src, tgt

model = Transformer(V, d_model=64, n_heads=4, d_ff=256, n_layers=2, max_len=16, pad=PAD)
STEPS, WARMUP = 1200, 100
opt = torch.optim.Adam(model.parameters(), lr=2e-3, betas=(0.9, 0.98))
# Doğrusal ısınma, ardından kosinüs azalması.
sched = torch.optim.lr_scheduler.LambdaLR(
    opt, lambda s: min(1.0, (s + 1) / WARMUP) * 0.5 * (1 + math.cos(math.pi * min(s, STEPS) / STEPS)))

model.train()
for step in range(STEPS):
    L = np.random.choice([4, 5, 6, 7])
    src, tgt = make_batch(64, L)
    tgt_in, tgt_out = tgt[:, :-1], tgt[:, 1:]
    logits = model(src, tgt_in)
    loss = F.cross_entropy(logits.reshape(-1, V), tgt_out.reshape(-1), ignore_index=PAD)
    opt.zero_grad(); loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    opt.step(); sched.step()
    if (step + 1) % 300 == 0:
        print(f"adım {step+1:4d}  kayıp = {loss.item():.3f}")

# Değerlendir (dropout kapalı)
model.eval()

@torch.no_grad()
def accuracy(L, n=200):
    src, tgt = make_batch(n, L)
    logits = model(src, tgt[:, :-1])
    preds = logits.argmax(-1)
    tgt_out = tgt[:, 1:]
    mask = (tgt_out != PAD)
    return ((preds == tgt_out) | ~mask).all(dim=1).float().mean().item()

print()
for L in [4, 5, 6, 7, 8, 10]:
    note = "" if L <= 7 else "   (eğitimde görülmemiş uzunluk)"
    print(f"  uzunluk {L:2d}: doğruluk = {accuracy(L):.3f}{note}")
print("\nSinüzoidal mutlak konum kodlamasıyla model eğitim uzunluklarında kusursuz, görülmemiş uzunluklarda başarısızdır;"
      "\nbu, 3. alıştırmadaki RoPE / ALiBi karşılaştırmasının çıkış noktasıdır.")

## 6. Nedensel maskeyi görselleştirmek

Kod çözücünün öz-dikkati ileriye bakmamalıdır. Maske üst üçgende `True`'dur — bu konumlar softmax'tan önce `-inf` alır.

In [ ]:
causal = torch.triu(torch.ones(8, 8, dtype=torch.bool), diagonal=1).numpy()
fig, ax = plt.subplots(figsize=(5, 5))
ax.imshow(~causal, cmap='Greens')
for i in range(8):
    for j in range(8):
        ax.text(j, i, '✓' if not causal[i, j] else '✗', ha='center', va='center',
                color='black' if not causal[i, j] else 'red', fontsize=12)
ax.set(title='Nedensel maske (✓ = dikkat izni var)', xlabel='anahtar konumu', ylabel='sorgu konumu')
ax.set_xticks(range(8)); ax.set_yticks(range(8))
plt.tight_layout(); plt.show()

## 7. RoPE — döner konum gömmesi

RoPE, gömmeye bir konum vektörü eklemek yerine $Q$ ve $K$'nin boyut çiftlerini konuma bağlı bir açıyla döndürür. Ortaya çıkan nokta çarpımı yalnızca göreli konum $i - j$'ye bağlıdır — tam istediğimiz özellik.

$m$ konumundaki bir 2B çift $(x_1, x_2)$ için:

$$R_\theta^{(m)} \begin{pmatrix} x_1 \\ x_2 \end{pmatrix} = \begin{pmatrix} \cos m\theta & -\sin m\theta \\ \sin m\theta & \cos m\theta \end{pmatrix} \begin{pmatrix} x_1 \\ x_2 \end{pmatrix}.$$

Aşağıdaki uygulama, çiftleri ardışık boyutlardan $(2i, 2i+1)$ değil, $i$ ve $i + d/2$ boyutlarından oluşturur (GPT-NeoX / LLaMA uygulamalarındaki "yarıya bölme" düzeni). İki düzen matematiksel olarak eşdeğerdir; yalnızca boyutların sırası farklıdır.

In [ ]:
def apply_rope(x, base=10000):
    # x: (..., T, d); d çift olmalı. Boyut i ile i + d/2 çifti döndürülür (yarıya bölme düzeni).
    T, d = x.size(-2), x.size(-1)
    assert d % 2 == 0
    half = d // 2
    inv_freq = 1.0 / (base ** (torch.arange(0, half).float() / half))
    pos = torch.arange(T).float().unsqueeze(1)               # (T, 1)
    freqs = pos * inv_freq.unsqueeze(0)                       # (T, half)
    cos = torch.cos(freqs); sin = torch.sin(freqs)
    x1, x2 = x[..., :half], x[..., half:]
    x_rot1 = x1 * cos - x2 * sin
    x_rot2 = x1 * sin + x2 * cos
    return torch.cat([x_rot1, x_rot2], dim=-1)

# Göreli konum özelliğini doğrula: Q ve K'yi kendi konumlarına göre döndürmek,
# Q·K'yi yalnızca (m - n)'ye bağımlı bırakır.
d = 16
q = torch.randn(d); k = torch.randn(d)
for (m1, n1), (m2, n2) in [((3, 1), (7, 5)), ((0, 2), (8, 10))]:
    Q1 = apply_rope(q.unsqueeze(0).expand(20, -1).unsqueeze(0))[0, m1]
    K1 = apply_rope(k.unsqueeze(0).expand(20, -1).unsqueeze(0))[0, n1]
    Q2 = apply_rope(q.unsqueeze(0).expand(20, -1).unsqueeze(0))[0, m2]
    K2 = apply_rope(k.unsqueeze(0).expand(20, -1).unsqueeze(0))[0, n2]
    print(f"m-n = {m1-n1}: Q·K = {(Q1@K1).item():+.4f}    "
          f"m-n = {m2-n2}: Q·K = {(Q2@K2).item():+.4f}    "
          f"eşit mi? {torch.allclose(Q1@K1, Q2@K2, atol=1e-4)}")

## 8. Alıştırmalar

1. **Neden $\sqrt{d_k}$?** Deneysel olarak gösterin: $d_k$ büyüdükçe ölçeklenmemiş skorların varyansı doğrusal olarak artar. Ölçekleme olmadan ortaya çıkan softmax neredeyse tek-sıcak (one-hot) olur, gradyanlar kaybolur ve eğitim durur.
2. **Ön-normalizasyon ve son-normalizasyon.** Her iki varyantı da uygulayın. Her birini aynı hiperparametrelerle eğitin. Eğitim kaybı eğrilerini çizdirin. Ön-normalizasyon daha pürüzsüz olmalı ve öğrenme oranı seçimine karşı daha hoşgörülü olmalıdır (Xiong vd., 2020). Isınmayı kaldırıp ikisini yeniden karşılaştırın.
3. **Uzunluğa genelleme için RoPE.** Sinüzoidal PE'yi RoPE ile değiştirin. 8 uzunluğuna kadar dizilerle eğitip 16 uzunluğunda değerlendirin. Sinüzoidal modelle karşılaştırın. Hangisi daha iyi genelleniyor? ALiBi'yi de deneyin.
4. **PyTorch ile eşleştirin.** Dikkatli bir ağırlık aktarımıyla yukarıdaki uygulama, `torch.nn.Transformer` çıktısıyla kayan nokta hassasiyetinde eşleşmelidir (`norm_first=True` kullanın). Bunu tek bir yığın üzerinde doğrulayın.

---

## Gelecek Hafta

Hafta 9 — Önceden eğitilmiş modeller. BERT, GPT, T5 — ön eğitimin üç felsefesi.